# Value-add tagging

Reads a staging export workbook, finds the field headers you highlighted **orange**, and
writes a **Value-add** column onto the `Fields` sheet with a `1` against every table/field
pair you marked.

## Why this resolves theme colours instead of matching a hex code

Excel does not store a highlight as `#F79646`. It stores *"theme colour 9, lightened 80%"*,
and what that renders as depends on the theme embedded in the workbook. So this notebook
reads `xl/theme/theme1.xml` out of the file and resolves the indices itself.

That is not defensive programming for its own sake. On the first run this workbook had
**three** header colours, and only one of them was the tag:

| Stored as | Resolves to | Headers | |
|---|---|---:|---|
| theme 9 (accent6) | `#F79646` orange | 626 | the value-add tags |
| theme 4 (accent1) | `#4F81BD` blue | 82 | something else |
| `2F4858` | slate | 739 | the exporter's own header colour |

Treating "anything that is not the standard colour" as a tag would have flagged 82 blue
fields you did not mark. Matching a hard-coded orange hex would have matched nothing at all,
because no cell in the file contains that string.

## Two other things this handles

**Sheet names are truncated at 31 characters** — Excel's limit. One table hits it:
`stg_team_recruiting_position_group` becomes the sheet `stg_team_recruiting_position_gr`.
Matching on the full name would drop that table's tags without saying so.

**Your annotated file is not overwritten by default.** The output goes to a new workbook,
because openpyxl rewrites a file rather than editing it, and your highlighting is the input
here. Set `OUT_FILENAME = FILENAME` if you would rather overwrite.


## Setup


In [ ]:
import re
import warnings
import zipfile
from copy import copy
from pathlib import Path


import pandas as pd
from IPython.display import HTML, display
from openpyxl import load_workbook

display(HTML("""
<style>
table.dataframe {
    border-collapse: collapse !important;
    border: 2px solid rgba(0,0,0,0.2) !important;
}
table.dataframe td, table.dataframe th {
    border: 1.5px solid rgba(0,0,0,0.15) !important;
}
table.dataframe thead th {
    background-color: #f0f0f0 !important;
    font-weight: bold !important;
}
</style>
"""))

## Configure

`FOLDER` is relative to the repository root, so this resolves the same wherever the notebook
is run from.


In [ ]:
# --- what to read ---------------------------------------------------------------------
FOLDER = "data/exports"
FILENAME = "staging_tables_20260901_1430.xlsx"

# --- what to write --------------------------------------------------------------------
# None derives '<name>_valueadd.xlsx' beside the input. Set to FILENAME to overwrite.
OUT_FILENAME = None

# --- which highlight counts -------------------------------------------------------------
# RESOLVED colours, not raw cell values: theme indices are turned into hex first, so this
# list is readable and portable. Run the scan below to see everything present in the file.
TAG_COLOURS = ["F79646"]        # orange (theme accent6)
# The colour the exporter paints every header, listed so the report can label it.
STANDARD_HEADER_RGB = "2F4858"

# --- sheet and column names ---------------------------------------------------------------
FIELDS_SHEET = "Fields"
SKIP_SHEETS = {"Index", FIELDS_SHEET}
VALUE_ADD_COLUMN = "Value-add"


def _repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


ROOT = _repo_root(Path.cwd())
IN_PATH = ROOT / FOLDER / FILENAME
OUT_PATH = (ROOT / FOLDER /
            (OUT_FILENAME or f"{Path(FILENAME).stem}_valueadd{Path(FILENAME).suffix}"))

assert IN_PATH.exists(), f"not found: {IN_PATH}"
print(f"in   {IN_PATH}")
print(f"out  {OUT_PATH}")
print(f"     {IN_PATH.stat().st_size / 1e6:.1f} MB")

## Resolve the workbook's theme

Excel's fill theme indices are not the order the colours appear in the XML — `lt1`/`dk1` and
`lt2`/`dk2` are swapped. Getting this wrong silently mislabels every colour, so the mapping
is written out rather than assumed.


In [ ]:
# Excel's theme index order for fills. NOT the XML order.
THEME_ORDER = ["lt1", "dk1", "lt2", "dk2", "accent1", "accent2", "accent3",
               "accent4", "accent5", "accent6", "hlink", "folHlink"]


def theme_palette(path):
    """{theme index: hex} read from the workbook's own theme.

    A slot can be an explicit srgbClr or a sysClr carrying lastClr — window text and window
    background are usually the latter, so both spellings are read.
    """
    with zipfile.ZipFile(path) as archive:
        theme = archive.read("xl/theme/theme1.xml").decode("utf-8")
    scheme = re.search(r"<a:clrScheme.*?</a:clrScheme>", theme, re.S).group(0)
    slots = re.findall(
        r'<a:(dk1|lt1|dk2|lt2|accent[1-6]|hlink|folHlink)>.*?'
        r'(?:srgbClr val="([0-9A-Fa-f]{6})"|sysClr[^>]*lastClr="([0-9A-Fa-f]{6})")',
        scheme, re.S)
    by_slot = {name: (a or b).upper() for name, a, b in slots}
    return {index: by_slot[slot] for index, slot in enumerate(THEME_ORDER)
            if slot in by_slot}


PALETTE = theme_palette(IN_PATH)
display(pd.DataFrame(
    [(i, THEME_ORDER[i], f"#{rgb}") for i, rgb in sorted(PALETTE.items())],
    columns=["Theme index", "Slot", "Hex"]))

## Scan every sheet for highlighted headers

Read-only mode: this pass needs row 1 of each sheet, and a full load of a 35 MB workbook
costs a minute and a couple of gigabytes.


In [ ]:
def header_colour(cell, palette):
    """(hex, tint) for a cell's fill, or (None, None) if it has none.

    Three spellings reach us and all three are real:
      rgb      written by the exporter, e.g. '002F4858' — the alpha prefix varies by writer,
               so only the last six digits are compared
      theme    written by Excel when you pick from the palette — resolved via the workbook's
               own theme, because the same index is a different colour in another theme
      indexed  the legacy palette; labelled rather than dropped, since a tag we cannot name
               is still a tag
    """
    fill = cell.fill
    if fill is None or fill.patternType is None:
        return None, None
    colour = fill.fgColor
    tint = round(colour.tint, 3) if colour.tint else 0.0
    if colour.type == "rgb" and isinstance(colour.rgb, str):
        return colour.rgb[-6:].upper(), tint
    if colour.type == "theme":
        return palette.get(colour.theme, f"theme-{colour.theme}"), tint
    if colour.type == "indexed":
        return f"indexed-{colour.indexed}", tint
    return None, None


def scan(path, palette):
    """(sheet, field, hex, tint) for every header that carries a fill."""
    workbook = load_workbook(path, read_only=True)
    try:
        rows = []
        for name in workbook.sheetnames:
            if name in SKIP_SHEETS:
                continue
            for cell in next(workbook[name].iter_rows(min_row=1, max_row=1), ()):
                if cell.value is None:
                    continue
                hex_value, tint = header_colour(cell, palette)
                if hex_value:
                    rows.append((name, str(cell.value), hex_value, tint))
        return pd.DataFrame(rows, columns=["Sheet", "Field", "Colour", "Tint"])
    finally:
        workbook.close()


headers = scan(IN_PATH, PALETTE)

# Every colour in the file, so a tag in a shade you did not expect is visible rather than
# quietly excluded. Tint is shown because Excel's palette rows are the same colour lightened.
summary = (headers.groupby(["Colour", "Tint"]).size().rename("Headers")
           .reset_index().sort_values("Headers", ascending=False))
summary["Selected"] = summary["Colour"].isin([c.upper() for c in TAG_COLOURS])
summary["Note"] = ["exporter default" if c == STANDARD_HEADER_RGB.upper() else ""
                   for c in summary["Colour"]]
display(summary.reset_index(drop=True))

### The tags this run will apply


In [ ]:
tagged = headers[headers["Colour"].isin([c.upper() for c in TAG_COLOURS])]
print(f"{len(tagged)} tagged header(s) across {tagged['Sheet'].nunique()} sheet(s)")

ignored = headers[(~headers["Colour"].isin([c.upper() for c in TAG_COLOURS]))
                  & (headers["Colour"] != STANDARD_HEADER_RGB.upper())]
if len(ignored):
    # Loud, not silent: a colour you meant as a tag and did not list is indistinguishable
    # from one you meant to ignore, and only you can tell them apart.
    print(f"\n{len(ignored)} header(s) carry a NON-STANDARD colour that is not in "
          f"TAG_COLOURS and will NOT be marked:")
    for colour, group in ignored.groupby("Colour"):
        print(f"  #{colour}: {len(group)} header(s), e.g. "
              f"{', '.join(group['Sheet'] + '.' + group['Field'])[:100]}")

display(tagged.sort_values(["Sheet", "Field"]).reset_index(drop=True).head(25))

## What a round trip will drop

openpyxl rewrites a workbook rather than editing it, and it does not understand every part
of the format. Anything it cannot parse is **silently discarded on save** — it emits a
warning buried in stderr and carries on.

This checks the file directly and says so up front, because "my data bars vanished" is a
bad thing to discover a week later. It is also the reason the default output is a new file
rather than an overwrite.


In [ ]:
def round_trip_losses(path):
    """Sheet-level features openpyxl will not carry through a save.

    Read from the file's XML rather than from openpyxl's own warnings: the warnings fire
    during load and say what was dropped, but not WHERE, and a sheet name is what makes the
    loss actionable.
    """
    losses = []
    with zipfile.ZipFile(path) as archive:
        names = [n for n in archive.namelist() if n.startswith("xl/worksheets/sheet")]
        for entry in names:
            body = archive.read(entry)
            for marker, label in ((b"<conditionalFormatting", "conditional formatting"),
                                  (b"<dataValidation", "data validation"),
                                  (b"<drawing ", "drawing or chart")):
                if marker in body:
                    losses.append((entry.rsplit("/", 1)[-1], label))
    return losses


losses = round_trip_losses(IN_PATH)
if losses:
    print("CAUTION — these will NOT survive the save:")
    for entry, label in losses:
        print(f"  {entry}: {label}")
    print("\nThe cell values, your header highlighting and the new column are all kept.")
    print(f"The original is untouched as long as OUT_FILENAME is not '{FILENAME}'.")
else:
    print("nothing in this workbook is at risk from the round trip")

## Apply the tags to the Fields sheet

This pass loads the workbook fully, which it must in order to write. Expect roughly a minute
and a couple of gigabytes for a 35 MB file.


In [ ]:
tags = {}
for row in tagged.itertuples():
    tags.setdefault(row.Sheet, set()).add(row.Field)

# Narrowly scoped: the losses were reported above, so the duplicate stderr noise is
# suppressed here. Any OTHER warning still surfaces.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*not supported and will be removed")
    workbook = load_workbook(IN_PATH)
fields = workbook[FIELDS_SHEET]

# FIND the header row rather than assuming it. The sheet carries a title and notes above the
# table, and hard-coding a row number breaks silently the next time a note is added.
header_row = next((cell.row for (cell,) in fields.iter_rows(min_col=1, max_col=1)
                   if str(cell.value).strip().lower() == "table"), None)
assert header_row, f"no 'Table' header found in the {FIELDS_SHEET} sheet"

columns = {str(c.value).strip(): c.column
           for c in next(fields.iter_rows(min_row=header_row, max_row=header_row))
           if c.value}
table_col, field_col = columns["Table"], columns["Field"]

# Reuse the column if this has already run, so a second pass corrects rather than appending
# a duplicate beside the first.
value_col = columns.get(VALUE_ADD_COLUMN, max(columns.values()) + 1)
template = next(fields.iter_rows(min_row=header_row, max_row=header_row))[0]
heading = fields.cell(header_row, value_col, VALUE_ADD_COLUMN)
heading.font, heading.fill = copy(template.font), copy(template.fill)

marked, seen_pairs = 0, set()
for row in range(header_row + 1, fields.max_row + 1):
    table = fields.cell(row, table_col).value
    field = fields.cell(row, field_col).value
    if not table or not field:
        continue
    # Sheet names are cut at Excel's 31-character limit, so compare the truncated form.
    key = (str(table)[:31], str(field))
    seen_pairs.add(key)
    if key[1] in tags.get(key[0], ()):
        fields.cell(row, value_col, 1)
        marked += 1
    else:
        # Blank, not 0 — a 1 against what you tagged was the ask, and a blank filters and
        # sorts just as well without asserting anything about the rest.
        fields.cell(row, value_col, None)

print(f"{len(seen_pairs):,} field rows scanned")
print(f"{marked} marked {VALUE_ADD_COLUMN} = 1")

# A tag matching no row is a real signal: a highlighted cell that is not a field name, or a
# sheet whose table is missing from the inventory.
unmatched = {(s, f) for s, fs in tags.items() for f in fs} - seen_pairs
if unmatched:
    print(f"\n{len(unmatched)} tagged header(s) matched NO row in {FIELDS_SHEET}:")
    for sheet_name, field in sorted(unmatched):
        print(f"  {sheet_name} . {field}")
else:
    print("every tagged header matched a field row")

## Save


In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
workbook.save(OUT_PATH)
workbook.close()
print(f"saved {OUT_PATH}  ({OUT_PATH.stat().st_size / 1e6:.1f} MB)")

## Verify what is on disk

Read the saved file back rather than trusting the counters above: those describe what the
code intended, this describes what was written.


In [ ]:
check = load_workbook(OUT_PATH, read_only=True)
rows = list(check[FIELDS_SHEET].iter_rows(values_only=True))
check.close()
head = next(i for i, r in enumerate(rows) if r and str(r[0]).strip().lower() == "table")
frame = pd.DataFrame(rows[head + 1:], columns=[str(c) for c in rows[head]])
frame = frame[frame["Table"].notna()]

flagged = frame[frame[VALUE_ADD_COLUMN] == 1]
print(f"{len(frame):,} field rows, {len(flagged)} flagged across "
      f"{flagged['Table'].nunique()} table(s)")
display(flagged.groupby("Table").size().rename("Value-add fields")
        .sort_values(ascending=False).to_frame().head(15))
display(flagged[["Table", "Field", "Data type", "Null %", "Cardinality"]]
        .reset_index(drop=True).head(25))